In [1]:
import pandas as pd

# set the folders that contain the manual segmenations and auto-segmantations
# Also, provide the dice_file that need to be updated with performance metrics. 
# The dice file will need to be prepared with colum format s.no, file_name, c3_slice_expert, c3_slice_auto
manual_seg_dir = '../data/raw-data/expert-segmentations/'
auto_seg_dir ='../data/val/output_segmentation/'
dice_file = '../data/Val-dice.csv'

# read the CSV files


df_infer = pd.read_csv(dice_file)

manual_seg_dir = '../data/raw-data/expert-segmentations/'
auto_seg_dir ='../data/val/output_segmentation/'


print(df_infer)


    Unnamed: 0             patient_id  C3_slice_expert  C3_slice_auto  \
0            0  mdacc_HNSCC-01-0216_C              192            193   
1            0  mdacc_HNSCC-01-0221_C              161            160   
2            0  mdacc_HNSCC-01-0231_C              188            188   
3            0  mdacc_HNSCC-01-0236_C              160            161   
4            0  mdacc_HNSCC-01-0241_C               47             47   
..         ...                    ...              ...            ...   
65           0  mdacc_HNSCC-01-0562_C              159            158   
66           0  mdacc_HNSCC-01-0567_C              175            170   
67           0  mdacc_HNSCC-01-0573_C               57             59   
68           0  mdacc_HNSCC-01-0579_C              178            179   
69           0  mdacc_HNSCC-01-0584_C              159            158   

    C3_slice_delta  muscle_dice  precsion  recall  Muscle_Area_Ground_Truth  \
0                0         0.96      0.99   

In [2]:
# Precision and Recall function definitions
def precision(gt, pr):
    TP = np.logical_and(gt, pr).sum()
    FP = pr[(pr==1) & (gt==0)].sum()
    deno = TP+FP
    if deno == 0:
        return np.NaN
    return TP/deno


def recall(gt, pr):
    TP = np.logical_and(gt, pr).sum()
    FN = gt[(gt==1) & (pr==0)].sum()
    deno = TP+FN
    if deno == 0:
        return np.NaN
    return TP/deno


In [3]:
#Generate the CSV file with performance evaluation metrics

from util.image_util.image_window import get_image_path_by_id,apply_window
from util.image_util.slice_array_from_nifty import get_C3_seg_array_by_id
from util.image_util.slice_area_density import get_c3_slice_area,get_c3_slice_density
import SimpleITK as sitk
import numpy as np
import pandas as pd
from pprint import pprint
from modeltrain.losses import dice_coef_multiclass_2D
from util.image_util.image_window import get_image_path_by_id


df_init = pd.DataFrame()
df_init_icc = pd.DataFrame()



#Calculate the Dice scores and save the data

for idx in range(df_infer.shape[0]):
# for idx in range(3):
    patient_id =df_infer.iloc[idx,1][:21]
    c3_slice_manual = df_infer.iloc[idx,-8]
    print(c3_slice_manual)
    c3_slice_auto = df_infer.iloc[idx,-7]
    print(c3_slice_auto)  
    
    
    muscle_manual_seg = get_C3_seg_array_by_id(patient_id,c3_slice_manual,manual_seg_dir)
    muscle_auto_seg = get_C3_seg_array_by_id(patient_id,c3_slice_auto,auto_seg_dir)

    muscle_dice = (2*np.sum(muscle_manual_seg*muscle_auto_seg))  \
                            /(np.sum(muscle_manual_seg)+np.sum(muscle_auto_seg))
    pre = precision(muscle_manual_seg,muscle_auto_seg)
    rec = recall(muscle_manual_seg, muscle_auto_seg)
    
    muscle_manual_area = get_c3_slice_area(patient_id,c3_slice_manual,manual_seg_dir)  

    
    muscle_auto_area = get_c3_slice_area(patient_id,c3_slice_auto,auto_seg_dir)  

    
    
    df_inter = pd.DataFrame({'patient_id':patient_id,
                                'C3_slice_expert':c3_slice_manual,
                                'C3_slice_auto':c3_slice_auto,
                                'C3_slice_delta':(c3_slice_manual - c3_slice_auto),
                                'muscle_dice':round(muscle_dice,2),
                                'precsion':round(pre,2),
                                'recall':round(rec,2),
                                'Expert_Muscle_Area':round(muscle_manual_area,2),
                                'Auto_Muscle_Area':round(muscle_auto_area,2)                                                     
                            },index=[0])
    
    ###Code block to build out dataframe for ICC Calculation
    if not (patient_id == 'mdacc_HNSCC-01-0623_C' or patient_id == 'mdacc_HNSCC-01-0613_C' ) :         
        df_infer_icc_m = pd.DataFrame({'patient_id':patient_id,'muscle_plot':'manual', 'muscle_area':round(muscle_manual_area, 2)},index=[0])
        df_init_icc = df_init_icc.append(df_infer_icc_m)
        df_infer_icc_a = pd.DataFrame({'patient_id':patient_id,'muscle_plot':'auto', 'muscle_area':round(muscle_auto_area, 2)},index=[0])
        df_init_icc = df_init_icc.append(df_infer_icc_a)        
                
    
    
    df_init = df_init.append(df_inter)
    df_init.to_csv(dice_file)
    print(idx+1,'th',patient_id,'dice saved')
    print(muscle_dice)
    if (muscle_dice< 0.8):print('muscle_dice',muscle_dice,patient_id)
    print()

192
193


/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:59: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_m)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:61: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_a)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:65: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init = df_init.append(df_inter)


1 th mdacc_HNSCC-01-0216_C dice saved
0.9472547728768927

161
160


/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:59: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_m)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:61: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_a)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:65: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init = df_init.append(df_inter)


2 th mdacc_HNSCC-01-0221_C dice saved
0.8790990599695964

188
188


...

/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:59: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_m)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:61: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_a)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:65: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init = df_init.append(df_inter)


68 th mdacc_HNSCC-01-0573_C dice saved
0.851864640883978

178
179


/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:59: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_m)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:61: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_a)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:65: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init = df_init.append(df_inter)


69 th mdacc_HNSCC-01-0579_C dice saved
0.9533153801246681

159
158
70 th mdacc_HNSCC-01-0584_C dice saved
0.9411331635939753



/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:59: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_m)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:61: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_a)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_21040/2612523919.py:65: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init = df_init.append(df_inter)


In [4]:
import pingouin as pg
print(df_init_icc)
icc = pg.intraclass_corr(data=df_init_icc, targets='patient_id', raters='muscle_plot', ratings='muscle_area' )
icc.set_index('Type')

               patient_id muscle_plot  muscle_area
0   mdacc_HNSCC-01-0216_C      manual        46.63
0   mdacc_HNSCC-01-0216_C        auto        43.90
0   mdacc_HNSCC-01-0221_C      manual        40.43
0   mdacc_HNSCC-01-0221_C        auto        36.42
0   mdacc_HNSCC-01-0231_C      manual        51.91
..                    ...         ...          ...
0   mdacc_HNSCC-01-0573_C        auto        43.81
0   mdacc_HNSCC-01-0579_C      manual        64.30
0   mdacc_HNSCC-01-0579_C        auto        62.30
0   mdacc_HNSCC-01-0584_C      manual        39.64
0   mdacc_HNSCC-01-0584_C        auto        38.08

[140 rows x 3 columns]


,Description,ICC,F,df1,df2,pval,CI95%
Type,,,,,,,
ICC1,Single raters absolute,0.915635,22.706562,69,70,1.168253e-29,"[0.87, 0.95]"
ICC2,Single random raters,0.917110,39.261661,69,69,5.331489e-37,"[0.64, 0.97]"
ICC3,Single fixed raters,0.950325,39.261661,69,69,5.331489e-37,"[0.92, 0.97]"
ICC1k,Average raters absolute,0.955960,22.706562,69,70,1.168253e-29,"[0.93, 0.97]"
ICC2k,Average random raters,0.956763,39.261661,69,69,5.331489e-37,"[0.78, 0.98]"
ICC3k,Average fixed raters,0.974530,39.261661,69,69,5.331489e-37,"[0.96, 0.98]"


/Users/yashravipati/miniforge3/envs/env_tf1/lib/python3.9/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package outdated is out of date. Your version is 0.2.1, the latest is 0.2.2.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(
/Users/yashravipati/miniforge3/envs/env_tf1/lib/python3.9/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.2, the latest is 0.5.3.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(
